## 3.1 Analyzing Token Frequency
- Goal: Determine the minimum threshold

In [1]:
import pandas as pd
from collections import Counter
train_v2_token = pd.read_parquet('data/processed/train_v2_tokenized.parquet')

token_frequency = Counter()
for tokens in train_v2_token['tokens']:
    token_frequency.update(set(tokens))


filtered_counter = Counter({word: count for word, count in token_frequency.items() if count == 4})
print(filtered_counter.most_common(5))
print(f"Number of Tokens: {len(token_frequency)}")
print(f"Number of Filtered Tokens: {len(filtered_counter)}")


[('checker', 4), ('alice', 4), ('node', 4), ('taxi', 4), ('"bust"', 4)]
Number of Tokens: 5144
Number of Filtered Tokens: 137


In [2]:
initial_total = len(token_frequency)
validation_v2 = pd.read_parquet("data/processed/validation_v2_tokenized.parquet")

validation_token_frequency = Counter()
for tokens in validation_v2['tokens']:
    validation_token_frequency.update(tokens)
total_validation_tokens = sum(validation_token_frequency.values())


## 3.2 - Looking at the Validation OOV Rate
- Validation Out of Vocaublary Rate
- Formula: Absent / All Validation Tokens
- I decided to choose `min_frequency = 4` vs choosing a `min_frequency = 3`. On the surface, choosing a min_freq of 3 seems great because it is able to remove 77.02% of rare rokens while maintaining a `1.89` OOV. Consider that the baseline (`min_freq = 0`)for the OOV was already 1.16 percent. By only increasing the Validation OOV Rate by only .15%, we are able to remove roughly 20% more than min_freq, while maintaining 97.96% of validation token occurrences.

In [3]:
print(f'Initial Starting Unique Token Amount: {initial_total}\n')
for min_freq in range(1, 6):
    print(f'min_freq = {min_freq}')

    removed_tokens_train = {word: count for word, count in token_frequency.items() if count < min_freq}
    removed_from_train = len(removed_tokens_train)
    vocab_size = initial_total - removed_from_train
    retained_vocab = {word: count for word, count in token_frequency.items() if count >= min_freq}
    out_of_vocab = 0

    for tokens in validation_v2['tokens']:
        for token in tokens:
            if token not in retained_vocab:
                out_of_vocab += 1

    print(f'Vocab Size Leftover: {vocab_size}')
    print(f'Removed (freqency < {min_freq}): {removed_from_train}')
    print(f'Percentage of Removed Tokens: {(removed_from_train / initial_total) * 100 :.2f}')
    print(f'Validation OOV Rate: {(out_of_vocab / total_validation_tokens) * 100:.2f}\n')

Initial Starting Unique Token Amount: 5144

min_freq = 1
Vocab Size Leftover: 5144
Removed (freqency < 1): 0
Percentage of Removed Tokens: 0.00
Validation OOV Rate: 1.16

min_freq = 2
Vocab Size Leftover: 1819
Removed (freqency < 2): 3325
Percentage of Removed Tokens: 64.64
Validation OOV Rate: 1.62

min_freq = 3
Vocab Size Leftover: 1182
Removed (freqency < 3): 3962
Percentage of Removed Tokens: 77.02
Validation OOV Rate: 1.89

min_freq = 4
Vocab Size Leftover: 943
Removed (freqency < 4): 4201
Percentage of Removed Tokens: 81.67
Validation OOV Rate: 2.04

min_freq = 5
Vocab Size Leftover: 806
Removed (freqency < 5): 4338
Percentage of Removed Tokens: 84.33
Validation OOV Rate: 2.19



## 3.3 - Investigating Rows with Comments
- Some of the rows had comments so in order to decide on how to move forward (remove or keep), I need to look at how important these comments are to the code
- Looked at rows with only comments to see if they were used for lables to determine if we can remove the comment tokens
- Conclusion: None of the rows in the examples were made up purely of just comments. The comment tokens that were found contribute to over 1,600 occurances consisting primarily of debug statements, notebook artfacts, encoding declarations, and natural language comments. Manual inspection showed that the five target labels were determined by the changes to the executable Python code rather than the comment contents. Comment tokens were tremoved from the tokenizer to reduce vocabulary size, while maintaining the represenation of executable code for the label.

In [4]:
from src.v2.comment_only_diff import is_comment_only

filtered_contains_comments = train_v2_token[train_v2_token['diff'].str.contains('#')]
print(f'Amount of rows that contain comments: {len(filtered_contains_comments)}')
for i in [1, 2, 3, 10, 20, 30, 50, 100, 150, 200, 300]:
    print(f"{i} ---- {filtered_contains_comments['diff'].iloc[i]} \n Top Level Label: {filtered_contains_comments['top_level_label'].iloc[i]}")

#Goal: return all of the lines that have comments only so this returns true
filtered_only_comments = train_v2_token.copy()
filtered_only_comments['only_comments'] = train_v2_token['diff'].apply(is_comment_only)
display(filtered_only_comments[filtered_only_comments['only_comments'] == True])

#Goal: find all of the unique comments strings in diff
unique_comments = Counter()
for row in filtered_contains_comments['diff']:
    lines = row.splitlines()
    for line in lines:
        line = line.replace(line[0], '', 1)
        line = line.strip()
        if not line:
            continue
        elif line.startswith('#'):
            unique_comments[line] += 1

for comment in unique_comments.most_common(10):
    print(comment)

print(len(unique_comments))
    

Amount of rows that contain comments: 1471
1 ---- - print(a)
+ #print(a) 
 Top Level Label: call
2 ---- - print(t)
+ # print(t) 
 Top Level Label: call
3 ---- - print(dp)
+ #print(dp) 
 Top Level Label: call
10 ---- + #162a
-     print(top_keta)
+     #print(top_keta) 
 Top Level Label: call
20 ---- - 		print(i)
+ 		#print(i) 
 Top Level Label: call
30 ---- -         print(sum_list)
+         # print(sum_list) 
 Top Level Label: call
50 ---- -       if s[i - 1] == s[j - 1] and LCSRe[i - 1][j - 1] < (j - i): 
+       if s[i - 1] == s[j - 1] and LCSRe[i - 1][j - 1] < j - i: 
-         else: 
+       else: 
-           LCSRe[i][j] = 0
+         LCSRe[i][j] = 0
+   #print(LCSRe) 
 Top Level Label: control_flow
100 ---- -     print(v)
+ # print(f"{pi} {qi}")
+ 
+  
 Top Level Label: call
150 ---- -   print(s)
+   #print(s) 
 Top Level Label: call
200 ---- - print(ans)
+ #print(ans) 
 Top Level Label: call
300 ---- - print(l)
+ #print(l) 
 Top Level Label: call


,diff,top_level_label,tokens,only_comments


('#print(s)', 30)
('#print(i)', 29)
('#print(a)', 20)
('#print(ans)', 19)
('#print(l)', 18)
('#print(dp)', 16)
("# '11111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111

## 3.4 Investigating Various Rows For Determining Minimum Threshold
- I saw tokens like `10000000000000` and I wanted to see it in context
- Upon inspecting an example, I found a top label of `call` where it had lots of indendation surrounding it. This could be potential noise for the model so I decided to change the tokenizer to reflect the indentation rather than just removing it.

In [5]:
contains_token = train_v2_token[train_v2_token['diff'].str.contains('gcds')]

print(len(contains_token['diff']))
for i in range(1):
    print(f'Top Level Label: {contains_token['top_level_label'].iloc[i]}')
    print(contains_token['diff'].iloc[i])
    print('------')


1
Top Level Label: expression
+ 
- for i in range(n + 1):
+ for i in range(n):
- 	gcds = max(gcds,(math.gcd(r[i],l[n - i])))
+ 	gcds = max(gcds,(math.gcd(r[i],l[n - i - 1])))
------


## 3.5 - Investigating the Indents Token
- This helped me decide whether or not I needed to keep certain indent tokens. I decided to keep them because they are useful context for the model to distinguish between indentation changes and real changes.

In [6]:
indents = {token: count for token, count in token_frequency.items() if token.startswith('<INDENT_')}
print(indents)


{'<INDENT_0>': 14887, '<INDENT_1>': 6149, '<INDENT_3>': 971, '<INDENT_2>': 2702, '<INDENT_4>': 261, '<INDENT_5>': 60, '<INDENT_6>': 12, '<INDENT_7>': 1, '<INDENT_9>': 1}


## 3.6 - Applying the `min_freq` and maintaining created Markers

In [7]:
vocabulary_after_applying_min = {token: count for token, count in token_frequency.items() if count >= 4 or token in ['<ADD>', '<DELETE>'] or token.startswith('<INDENT_')}
print(f'Size of Vocabulary: {len(vocabulary_after_applying_min)}')

Size of Vocabulary: 945


## 3.7 - Building the Vocabulary

In [8]:
from src.v2.ids_and_tokens import create_token_to_id, create_id_to_token
import json
final_retained_vocab = {word: count for word, count in token_frequency.items() if count >= 4 or word.startswith('<INDENT_') or word in ('<ADD>', '<DELETE')} #kept all indents
token_to_id = create_token_to_id(final_retained_vocab)
id_to_token = create_id_to_token(token_to_id)

with open('data/processed/token_to_id.json', 'w', encoding='utf-8') as file:
    json.dump(token_to_id, file, indent=2, ensure_ascii=False)

with open('data/processed/id_to_token.json', 'w', encoding='utf-8') as file:
    json.dump(id_to_token, file, indent=2, ensure_ascii=False)

print(f'Amount of tokens after min_freq: {len(final_retained_vocab)}')
print(f'Amount of tokens after token to id: {len(token_to_id)}') #should have +2 (<PAD> and <UNK>)


Amount of tokens after min_freq: 945
Amount of tokens after token to id: 947
